# HW4 Starter: Complex Neural Network Architecture

This notebook is a starting point for Homework 4. The exact assignment is not known yet, so the goal is to prepare a defensible neural-network direction that can be adapted later.

**Recommended architecture:** a **tabular residual neural network**, also called a **ResNet-style MLP**.

Why this is the best first choice for this project:

- The current HW2/HW3 dataset is mostly **tabular**: circuit features, compiler metadata, and hardware calibration/noise features.
- A normal CNN expects local spatial structure, such as pixels in an image. These feature columns do not naturally form an image.
- A residual MLP still counts as a complex neural architecture because it uses deep layers, skip connections, normalization, dropout, and nonlinear representation learning.
- It fits the thesis methodology better than forcing a CNN onto non-image data.

If the HW4 assignment explicitly requires CNN or ResNet wording, the safest explanation is: **we use the core ResNet idea, residual skip connections, adapted to tabular data.**

## 1. Current Task Assumption

Until the HW4 assignment is more specific, this notebook assumes the same task as the previous benchmark:

- **Input:** pre-run circuit, compiler, and hardware features
- **Target:** `reliability`
- **Task type:** supervised regression
- **Primary metric:** validation MAE
- **Leakage rule:** do not use outcome-derived columns such as `fidelity`, `error_rate`, counts payloads, or mitigation outputs as model inputs

TODO when HW4 is published:

- Confirm whether the target remains `reliability` or changes to fault-type classification.
- Confirm whether a specific architecture family is required.
- Confirm whether the assignment expects PyTorch, TensorFlow/Keras, or sklearn-style code.
- Confirm whether the final evaluation should use only validation data or also the held-out test split.

## 2. Architecture Choice

| Architecture | Fit for this dataset | Recommendation |
| --- | --- | --- |
| Plain MLP | Good baseline neural network for tabular features | Useful baseline |
| CNN | Weak fit unless features are converted into meaningful sequences or grids | Avoid unless required |
| Image ResNet | Weak fit for raw tabular columns | Avoid unless data becomes image-like |
| **Tabular ResNet / Residual MLP** | **Strong fit: deep model with skip connections for tabular data** | **Recommended** |
| FT-Transformer / TabTransformer | Strong but heavier and more complex | Good future upgrade |

The residual MLP is the ideal compromise: it is complex enough for HW4, honest about the data structure, and still relatively easy to explain in a thesis.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path('data/hw2')

np.random.seed(RANDOM_STATE)

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train.parquet')
validation = pd.read_parquet(DATA_DIR / 'validation.parquet')
test = pd.read_parquet(DATA_DIR / 'test.parquet')
feature_policy = json.loads((DATA_DIR / 'feature_policy.json').read_text(encoding='utf-8'))

print(f'Train rows: {len(train):,}')
print(f'Validation rows: {len(validation):,}')
print(f'Test rows, reserved for final evaluation: {len(test):,}')
print(f'Target column: {feature_policy["target_column"]}')
print(f'Allowed feature columns: {len(feature_policy["allowed_feature_columns"])}')

In [ ]:
target_column = feature_policy['target_column']
feature_columns = feature_policy['allowed_feature_columns']

X_train = train[feature_columns]
y_train = train[target_column].astype('float32')

X_validation = validation[feature_columns]
y_validation = validation[target_column].astype('float32')

categorical_columns = X_train.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

print(f'Numeric columns: {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)}')

## 3. Train-Only Preprocessing

The neural network should not learn preprocessing from validation or test rows. The imputer, encoder, and scaler are fitted only on the training split. This follows the same leakage-safe logic as the earlier benchmark notebook.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            categorical_columns,
        ),
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_columns,
        ),
    ],
    remainder='drop',
)

X_train_nn = preprocessor.fit_transform(X_train).astype('float32')
X_validation_nn = preprocessor.transform(X_validation).astype('float32')

print(f'Neural-network input shape: {X_train_nn.shape}')

## 4. Residual MLP Design

A residual block learns a correction to its input instead of replacing the representation completely:

`output = input + learned_transformation(input)`

This helps deeper networks train more reliably because information can flow through skip connections. For this project, the architecture is:

1. Input projection from engineered features into a hidden representation.
2. Several residual blocks with linear layers, batch normalization, ReLU, and dropout.
3. A final regression head that predicts one value: `reliability`.

TODO later: tune hidden size, number of residual blocks, dropout, learning rate, batch size, and early stopping patience.

In [ ]:
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch is not installed in this kernel yet.')
    print('Install it later if HW4 allows PyTorch, for example: pip install torch')

In [ ]:
if TORCH_AVAILABLE:
    class ResidualBlock(nn.Module):
        def __init__(self, width: int, dropout: float = 0.10):
            super().__init__()
            self.block = nn.Sequential(
                nn.Linear(width, width),
                nn.BatchNorm1d(width),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(width, width),
                nn.BatchNorm1d(width),
            )
            self.activation = nn.ReLU()

        def forward(self, x):
            return self.activation(x + self.block(x))


    class TabularResNetRegressor(nn.Module):
        def __init__(self, input_dim: int, width: int = 128, blocks: int = 3, dropout: float = 0.10):
            super().__init__()
            self.input_projection = nn.Sequential(
                nn.Linear(input_dim, width),
                nn.BatchNorm1d(width),
                nn.ReLU(),
            )
            self.residual_blocks = nn.Sequential(
                *[ResidualBlock(width=width, dropout=dropout) for _ in range(blocks)]
            )
            self.output_head = nn.Linear(width, 1)

        def forward(self, x):
            x = self.input_projection(x)
            x = self.residual_blocks(x)
            return self.output_head(x).squeeze(-1)


    model = TabularResNetRegressor(input_dim=X_train_nn.shape[1])
    print(model)

## 5. Training Skeleton

This is intentionally a starter training loop. Once the exact HW4 requirements are known, continue from here by adding:

- baseline comparison against Ridge/XGBoost/RandomForest or the previous HW notebook results,
- early stopping based on validation MAE,
- plots of training loss and validation MAE,
- final written interpretation of whether the complex neural network actually improves the result.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'r2': r2_score(y_true, y_pred),
    }


if TORCH_AVAILABLE:
    torch.manual_seed(RANDOM_STATE)

    train_dataset = TensorDataset(
        torch.tensor(X_train_nn),
        torch.tensor(y_train.to_numpy(dtype='float32')),
    )
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    model = TabularResNetRegressor(input_dim=X_train_nn.shape[1], width=128, blocks=3, dropout=0.10)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    epochs = 10
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = loss_fn(predictions, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch_X)

        model.eval()
        with torch.no_grad():
            validation_predictions = model(torch.tensor(X_validation_nn)).numpy()

        metrics = regression_metrics(y_validation, validation_predictions)
        print(
            f'Epoch {epoch:02d} | '
            f'train_mse={running_loss / len(train_dataset):.6f} | '
            f'validation_mae={metrics["mae"]:.6f} | '
            f'validation_rmse={metrics["rmse"]:.6f} | '
            f'validation_r2={metrics["r2"]:.4f}'
        )

## 6. Interpretation Template

Use this section after training results exist.

**Model selected:** Tabular residual neural network.

**Reason:** The dataset is tabular, so a ResNet-style MLP is a better architectural match than an image CNN. The residual connections make the model deeper while reducing optimization difficulty.

**Scientific caution:** A more complex neural network is not automatically better. The final conclusion should compare it against the previous simple baseline and explain whether the extra complexity is justified by validation performance.

**TODO results:** Add validation MAE, RMSE, and R2 after running the notebook.

**TODO conclusion:** Decide whether the residual MLP improves over the previous benchmark enough to be useful for the thesis.